# DoubleMambaUNet training on Kvasir-SEG

Adapted from the single-network VM-UNet training notebook. Model, loss, training loop, and evaluation are updated for `DoubleMambaUNet`'s 2-channel `(Output1, Output2)` prediction -- everything else (env setup, dataset, metrics) is unchanged from the original notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# PyTorch 2.3.1 + CUDA 12.1 -- has pre-built mamba_ssm wheels
!pip install torch==2.3.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu121 -q

In [ ]:
!pip install torchvision==0.18.1 -q
!pip install transformers==4.40.0 -q
!pip install tokenizers==0.19.1 -q
!pip install torchmetrics==1.4.0 -q
!pip install medpy -q
!pip install timm einops albumentations -q

In [ ]:
import torch
print(torch.version.cuda)
print(torch.__version__)

In [ ]:
!pip install causal-conv1d==1.4.0 -q
!pip install mamba-ssm==2.2.2 -q

In [ ]:
import torch
from mamba_ssm.ops.selective_scan_interface import selective_scan_fn
print("mamba_ssm loaded successfully!")
print(torch.__version__)   # should be 2.3.1+cu121

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import random
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torchmetrics
from torchmetrics import JaccardIndex, F1Score, Accuracy, Precision, Recall, Specificity
from medpy.metric import binary

# For reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Load DoubleMambaUNet source

Unlike the original notebook, there's no `.vmamba import` fix needed here -- `double_mamba_unet.py` already uses an absolute `from vmamba import (...)` (see its header), so it drops in as-is once `vmamba.py` sits next to it.

Upload `vmamba.py` and `double_mamba_unet.py` to the same Drive folder you were already using for VMUNet (`VMUNet_files/` below, or point `drive_source` at wherever you keep them).

In [ ]:
from google.colab import drive
import shutil, os

drive_source = '/content/drive/MyDrive/VMUNet_files/'

for file in ['vmamba.py', 'double_mamba_unet.py']:
    src = os.path.join(drive_source, file)
    dst = os.path.join('/content', file)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"loaded: {file}")
    else:
        print(f"MISSING: {file}  (expected at {src})")

import sys
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

In [ ]:
from double_mamba_unet import DoubleMambaUNet
print("DoubleMambaUNet imported successfully!")

In [ ]:
data_dir = "/content/drive/MyDrive/Kvasir-SEG"
image_dir = os.path.join(data_dir, "images")
mask_dir = os.path.join(data_dir, "masks")

IMG_HEIGHT = 256
IMG_WIDTH = 256
# DoubleMambaUNet holds two full Mamba encoder-decoder stacks in memory at once --
# halve the batch size vs. the single-network VM-UNet run as a starting point,
# and raise it back up if your GPU has headroom.
batch_size = 4
learning_rate = 1e-3
num_epochs = 100
patience = 40

os.makedirs("models", exist_ok=True)
os.makedirs("logs", exist_ok=True)
os.makedirs("results", exist_ok=True)

# Get all image and mask files
image_files = sorted(os.listdir(image_dir))
mask_files = sorted(os.listdir(mask_dir))
image_paths = [os.path.join(image_dir, f) for f in image_files]
mask_paths = [os.path.join(mask_dir, f) for f in mask_files]

# Split
from sklearn.model_selection import train_test_split
temp = list(zip(image_paths, mask_paths))
train_val, test = train_test_split(temp, test_size=0.15, random_state=42)
train, val = train_test_split(train_val, test_size=0.15/0.85, random_state=42)

print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

In [ ]:
class KvasirDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = cv2.imread(self.image_paths[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)
        mask = (mask > 0).astype(np.uint8)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
        else:
            image = torch.from_numpy(image).permute(2,0,1).float() / 255.0
            mask = torch.from_numpy(mask).long()
        return image, mask

# Transforms
train_transform = A.Compose([
    A.Resize(IMG_HEIGHT, IMG_WIDTH),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_HEIGHT, IMG_WIDTH),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

train_dataset = KvasirDataset(*zip(*train), transform=train_transform)
val_dataset = KvasirDataset(*zip(*val), transform=val_transform)
test_dataset = KvasirDataset(*zip(*test), transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

## Deep-supervision loss for the two outputs

`DoubleMambaUNet` returns a `(N, 2, H, W)` tensor: channel 0 is `Output1` (Network 1's intermediate mask), channel 1 is `Output2` (Network 2's final mask). `DoubleOutputLoss` applies the same Dice+BCE loss used for VM-UNet to each channel against the single ground-truth mask, then combines them -- this is deep supervision, matching DoubleU-Net's own two-mask design. `output2_weight` defaults to 0.5/0.5 (equal weight, matching the reference DoubleU-Net training setup); raise it if you want to push the network harder toward getting the *final* mask right specifically.

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self, dice_weight=0.5, bce_weight=0.5):
        super().__init__()
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.bce = nn.BCELoss()   # expects probabilities

    def forward(self, inputs, targets):
        # inputs: (N,1,H,W) or (N,H,W) probabilities in [0,1]
        # targets: (N,H,W) long (0/1)
        inputs = inputs.squeeze(1) if inputs.dim() == 4 else inputs  # (N,H,W)
        targets = targets.float()

        smooth = 1e-5
        intersection = (inputs * targets).sum()
        dice_loss = 1 - (2. * intersection + smooth) / (inputs.sum() + targets.sum() + smooth)

        bce_loss = self.bce(inputs, targets)

        return self.dice_weight * dice_loss + self.bce_weight * bce_loss


class DoubleOutputLoss(nn.Module):
    """Applies DiceBCELoss to Output1 and Output2 separately against the
    same mask, then combines with configurable weights (deep supervision)."""
    def __init__(self, output1_weight=0.5, output2_weight=0.5,
                 dice_weight=0.5, bce_weight=0.5):
        super().__init__()
        self.output1_weight = output1_weight
        self.output2_weight = output2_weight
        self.base_loss = DiceBCELoss(dice_weight, bce_weight)

    def forward(self, outputs, targets):
        # outputs: (N,2,H,W) -- [:,0]=Output1, [:,1]=Output2
        output1 = outputs[:, 0:1]
        output2 = outputs[:, 1:2]
        loss1 = self.base_loss(output1, targets)
        loss2 = self.base_loss(output2, targets)
        return self.output1_weight * loss1 + self.output2_weight * loss2

criterion = DoubleOutputLoss().to(device)

In [ ]:
from double_mamba_unet import DoubleMambaUNet

model = DoubleMambaUNet(
    in_chans=3,
    num_classes=1,
    depths=(2, 2, 2, 2),
    depths_decoder=(2, 2, 2, 2),
    drop_path_rate=0.2,
).to(device)

optimizer = optim.Adam(model.parameters(), lr=learning_rate)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)

print(f"Total parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

In [ ]:
jaccard = JaccardIndex(task='binary').to(device)
dice = F1Score(task='binary').to(device)
accuracy = Accuracy(task='binary').to(device)
precision = Precision(task='binary').to(device)
recall = Recall(task='binary').to(device)
specificity = Specificity(task='binary').to(device)

def compute_hausdorff(pred_mask, true_mask, spacing=(1,1)):
    pred_binary = pred_mask.astype(np.uint8)
    true_binary = true_mask.astype(np.uint8)
    if np.sum(pred_binary) == 0 or np.sum(true_binary) == 0:
        return np.nan, np.nan
    hd = binary.hd(pred_binary, true_binary, voxelspacing=spacing)
    hd95 = binary.hd95(pred_binary, true_binary, voxelspacing=spacing)
    return hd, hd95

## Train / validate

Same structure as the VM-UNet notebook. The only change: `outputs` is now `(N,2,H,W)`, loss uses the full 2-channel tensor via `DoubleOutputLoss`, and metrics/predictions are computed from `Output2` (`outputs[:, 1:2]`) -- the final mask, matching how DoubleU-Net itself is evaluated.

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total_pixels = 0

    for images, masks in tqdm(dataloader, desc='Training'):
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)          # (N,2,H,W) -- [Output1, Output2]
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        final_output = outputs[:, 1:2]   # Output2 -- the final prediction
        preds = (final_output > 0.5).squeeze(1).long()
        correct += (preds == masks).sum().item()
        total_pixels += masks.numel()

    avg_loss = total_loss / len(dataloader)
    avg_acc = correct / total_pixels
    return avg_loss, avg_acc

def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    jaccard.reset()
    dice.reset()
    accuracy.reset()
    precision.reset()
    recall.reset()
    specificity.reset()
    hd_list, hd95_list = [], []

    with torch.no_grad():
        for images, masks in tqdm(dataloader, desc='Validation'):
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)      # (N,2,H,W)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

            final_output = outputs[:, 1:2]   # Output2
            preds = (final_output > 0.5).squeeze(1).long()

            jaccard.update(preds, masks)
            dice.update(preds, masks)
            accuracy.update(preds, masks)
            precision.update(preds, masks)
            recall.update(preds, masks)
            specificity.update(preds, masks)

            for i in range(images.size(0)):
                pred_np = preds[i].cpu().numpy()
                mask_np = masks[i].cpu().numpy()
                hd, hd95 = compute_hausdorff(pred_np, mask_np)
                hd_list.append(hd)
                hd95_list.append(hd95)

    avg_loss = total_loss / len(dataloader)
    metrics = {
        'loss': avg_loss,
        'val_accuracy': accuracy.compute().cpu().item(),
        'jaccard': jaccard.compute().cpu().item(),
        'dice': dice.compute().cpu().item(),
        'precision': precision.compute().cpu().item(),
        'recall': recall.compute().cpu().item(),
        'specificity': specificity.compute().cpu().item(),
        'hd_mean': np.nanmean(hd_list),
        'hd95_mean': np.nanmean(hd95_list)
    }
    return metrics

In [ ]:
log_file = "logs/training_log.csv"
csv_columns = ['epoch', 'train_loss', 'train_accuracy', 'val_loss', 'val_accuracy',
               'jaccard', 'dice', 'precision', 'recall', 'specificity',
               'hd_mean', 'hd95_mean']

if not os.path.exists(log_file):
    pd.DataFrame(columns=csv_columns).to_csv(log_file, index=False)

best_val_loss = float('inf')
patience_counter = 0
history = defaultdict(list)

for epoch in range(1, num_epochs+1):
    print(f"\nEpoch {epoch}/{num_epochs}")

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_metrics = validate_one_epoch(model, val_loader, criterion, device)

    scheduler.step(val_metrics['loss'])

    history['epoch'].append(epoch)
    history['train_loss'].append(train_loss)
    history['train_accuracy'].append(train_acc)
    history['val_loss'].append(val_metrics['loss'])
    history['val_accuracy'].append(val_metrics['val_accuracy'])
    history['jaccard'].append(val_metrics['jaccard'])
    history['dice'].append(val_metrics['dice'])
    history['precision'].append(val_metrics['precision'])
    history['recall'].append(val_metrics['recall'])
    history['specificity'].append(val_metrics['specificity'])
    history['hd_mean'].append(val_metrics['hd_mean'])
    history['hd95_mean'].append(val_metrics['hd95_mean'])

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_metrics['loss']:.4f}")
    print(f"Train Acc:  {train_acc:.4f} | Val Acc:  {val_metrics['val_accuracy']:.4f}")
    print(f"Jaccard: {val_metrics['jaccard']:.4f} | Dice: {val_metrics['dice']:.4f}")
    print(f"HD: {val_metrics['hd_mean']:.2f} | HD95: {val_metrics['hd95_mean']:.2f}")

    row = {col: history[col][-1] for col in csv_columns}
    pd.DataFrame([row]).to_csv(log_file, mode='a', header=False, index=False)

    if val_metrics['loss'] < best_val_loss:
        best_val_loss = val_metrics['loss']
        torch.save(model.state_dict(), 'models/best_doublemambaunet.pth')
        print("Best model saved!")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

torch.save(model.state_dict(), 'models/final_doublemambaunet.pth')

In [ ]:
def plot_individual_curves(history):
    os.makedirs("results", exist_ok=True)
    epochs = history.get('epoch', [])
    if len(epochs) == 0:
        print("No epoch data found in history. Nothing to plot.")
        return

    train_loss = history.get('train_loss', [])
    val_loss = history.get('val_loss', history.get('loss', []))

    if len(train_loss) == len(epochs) and len(val_loss) == len(epochs):
        plt.figure(figsize=(8,6))
        plt.plot(epochs, train_loss, label='Train Loss')
        plt.plot(epochs, val_loss, label='Val Loss')
        plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss Curves')
        plt.legend(); plt.grid(True); plt.tight_layout()
        plt.savefig('results/loss_curve.png', dpi=150); plt.show()
        print("Saved: results/loss_curve.png")
    else:
        print("Skipping loss plot - data length mismatch or missing.")

    train_acc = history.get('train_accuracy', [])
    val_acc = history.get('val_accuracy', [])
    if len(train_acc) == len(epochs) and len(val_acc) == len(epochs):
        plt.figure(figsize=(8,6))
        plt.plot(epochs, train_acc, label='Train Accuracy')
        plt.plot(epochs, val_acc, label='Val Accuracy')
        plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('Accuracy Curves')
        plt.legend(); plt.grid(True); plt.tight_layout()
        plt.savefig('results/accuracy_curve.png', dpi=150); plt.show()
        print("Saved: results/accuracy_curve.png")
    else:
        print("Skipping accuracy plot - data missing.")

    jaccard_h = history.get('jaccard', [])
    dice_h = history.get('dice', [])
    if len(jaccard_h) == len(epochs) and len(dice_h) == len(epochs):
        plt.figure(figsize=(8,6))
        plt.plot(epochs, jaccard_h, label='Jaccard (IoU)')
        plt.plot(epochs, dice_h, label='Dice')
        plt.xlabel('Epoch'); plt.ylabel('Score'); plt.title('Jaccard and Dice')
        plt.legend(); plt.grid(True); plt.tight_layout()
        plt.savefig('results/jaccard_dice_curve.png', dpi=150); plt.show()
        print("Saved: results/jaccard_dice_curve.png")
    else:
        print("Skipping Jaccard/Dice plot - data missing.")

    precision_h = history.get('precision', [])
    recall_h = history.get('recall', [])
    specificity_h = history.get('specificity', [])
    if all(len(x) == len(epochs) for x in [precision_h, recall_h, specificity_h]):
        plt.figure(figsize=(8,6))
        plt.plot(epochs, precision_h, label='Precision')
        plt.plot(epochs, recall_h, label='Recall')
        plt.plot(epochs, specificity_h, label='Specificity')
        plt.xlabel('Epoch'); plt.ylabel('Score'); plt.title('Precision, Recall, Specificity')
        plt.legend(); plt.grid(True); plt.tight_layout()
        plt.savefig('results/precision_recall_specificity.png', dpi=150); plt.show()
        print("Saved: results/precision_recall_specificity.png")
    else:
        print("Skipping precision/recall/specificity plot - data missing.")

    hd_mean_h = history.get('hd_mean', [])
    hd95_mean_h = history.get('hd95_mean', [])
    if len(hd_mean_h) == len(epochs) and len(hd95_mean_h) == len(epochs):
        plt.figure(figsize=(8,6))
        plt.plot(epochs, hd_mean_h, label='HD')
        plt.plot(epochs, hd95_mean_h, label='HD95')
        plt.xlabel('Epoch'); plt.ylabel('Distance'); plt.title('Hausdorff Distance')
        plt.legend(); plt.grid(True); plt.tight_layout()
        plt.savefig('results/hausdorff_curve.png', dpi=150); plt.show()
        print("Saved: results/hausdorff_curve.png")
    else:
        print("Skipping Hausdorff plot - data missing.")

In [ ]:
plot_individual_curves(history)

In [ ]:
# Load best model
model.load_state_dict(torch.load('/content/models/best_doublemambaunet.pth'))
model.eval()

jaccard.reset(); dice.reset(); accuracy.reset()
precision.reset(); recall.reset(); specificity.reset()
hd_list = []
hd95_list = []
test_loss_total = 0

with torch.no_grad():
    for images, masks in tqdm(test_loader, desc='Testing'):
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)                 # (N,2,H,W)
        loss = criterion(outputs, masks)
        test_loss_total += loss.item()

        final_output = outputs[:, 1:2]          # Output2 -- final prediction
        preds = (final_output > 0.5).squeeze(1).long()

        jaccard.update(preds, masks)
        dice.update(preds, masks)
        accuracy.update(preds, masks)
        precision.update(preds, masks)
        recall.update(preds, masks)
        specificity.update(preds, masks)

        for i in range(images.size(0)):
            pred_np = preds[i].cpu().numpy()
            mask_np = masks[i].cpu().numpy()
            hd, hd95 = compute_hausdorff(pred_np, mask_np)
            hd_list.append(hd)
            hd95_list.append(hd95)

avg_test_loss = test_loss_total / len(test_loader)

jaccard_score = jaccard.compute().cpu().item()
dice_score = dice.compute().cpu().item()
acc_score = accuracy.compute().cpu().item()
prec_score = precision.compute().cpu().item()
rec_score = recall.compute().cpu().item()
spec_score = specificity.compute().cpu().item()
hd_mean = np.nanmean(hd_list)
hd95_mean = np.nanmean(hd95_list)

print("\n===== Test Set Results (Output2 / final prediction) =====")
print(f"Test Loss: {avg_test_loss:.4f}")
print(f"Accuracy: {acc_score:.4f}")
print(f"Jaccard: {jaccard_score:.4f}")
print(f"Dice: {dice_score:.4f}")
print(f"Precision: {prec_score:.4f}")
print(f"Recall: {rec_score:.4f}")
print(f"Specificity: {spec_score:.4f}")
print(f"HD: {hd_mean:.2f}")
print(f"HD95: {hd95_mean:.2f}")

test_results = pd.DataFrame([{
    'loss': avg_test_loss,
    'accuracy': acc_score,
    'jaccard': jaccard_score,
    'dice': dice_score,
    'precision': prec_score,
    'recall': rec_score,
    'specificity': spec_score,
    'hd': hd_mean,
    'hd95': hd95_mean
}])
test_results.to_csv('results/test_metrics.csv', index=False)
print("Test metrics saved to results/test_metrics.csv")

## Visualize predictions

Extended to 4 columns -- Image, Ground Truth, Output1, Output2 -- so you can see the qualitative improvement from Network 1's intermediate mask to Network 2's refined final mask, the same comparison DoubleU-Net's own paper highlights as the motivation for the two-network design.

In [ ]:
def visualize_test_predictions(model, dataset, num_samples=3):
    model.eval()
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, num_samples*4))
    indices = random.sample(range(len(dataset)), num_samples)

    for row, idx in enumerate(indices):
        img, mask = dataset[idx]
        img_tensor = img.unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(img_tensor)                                   # (1,2,H,W)
            pred1 = (output[:, 0:1] > 0.5).squeeze().cpu().numpy().astype(np.uint8)
            pred2 = (output[:, 1:2] > 0.5).squeeze().cpu().numpy().astype(np.uint8)

        img_np = img.permute(1,2,0).cpu().numpy()
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img_np = img_np * std + mean
        img_np = np.clip(img_np, 0, 1)

        mask_np = mask.cpu().numpy()

        axes[row,0].imshow(img_np); axes[row,0].set_title('Image'); axes[row,0].axis('off')
        axes[row,1].imshow(mask_np, cmap='gray'); axes[row,1].set_title('Ground Truth'); axes[row,1].axis('off')
        axes[row,2].imshow(pred1, cmap='gray'); axes[row,2].set_title('Output1 (Network 1)'); axes[row,2].axis('off')
        axes[row,3].imshow(pred2, cmap='gray'); axes[row,3].set_title('Output2 (final)'); axes[row,3].axis('off')

    plt.tight_layout()
    plt.savefig('results/test_predictions.png', dpi=150)
    plt.show()

visualize_test_predictions(model, test_dataset, num_samples=3)

In [ ]:
# Replace with your repo URL
remote_url = "git@github.com:Amir-Rouhbakhsh/mamba-models.git"

!git remote add origin {remote_url}

# Use a personal access token for authentication
# You will be prompted for username and password (use token as password)
!git push -u origin main